In [ ]:
from annotator import Annotator
from datasets import AGRDataset
from taxa import TaxonField, taxon_mapper

from pathlib import Path
import pandas as pd

In [ ]:
# init annotator with cache directory
ann = Annotator(Path("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache"))

In [ ]:
# Optional one-time bulk pull: orthology -> cache/bulk/orthology.parquet (the .tsv.gz is discarded).
await ann.download(AGRDataset.ORTHOLOGY)

In [ ]:
normalize = await ann.annotate(["TP53", "BRCA1", "EGFR", "RPA1", "CELSR3", "NOTAGENE", "C2orf40"], limit=5, case_insensitive=False)
normalize

In [ ]:
orth = await ann.get_orthologs(["TP53", "BRCA1", "EGFR", "RPA1", "CELSR3", "NOTAGENE"], taxon="human", limit=2)
orth["Gene2SpeciesTaxonCommonName"] = orth["Gene2SpeciesTaxonID"].map(taxon_mapper(TaxonField.COMMON_NAME))
orth

In [ ]:
# Filter-then-requery: pull the mouse orthologs back out as a fresh gene list.
mouse_ids = orth.loc[orth.Gene2SpeciesTaxonCommonName == "mouse", "Gene2ID"].dropna().unique().tolist()
mouse_ids

In [ ]:
# Per-gene API path (phenotypes have no bulk TSV): paginated + cached under cache/api/phenotypes/.
pheno = await ann.annotate(mouse_ids, AGRDataset.PHENOTYPES, taxon="mouse")
pheno

In [ ]:
brainspan_path = "/nfs/xing_lab/lab02/Data_Raw/Xiaolong/HumanCommon/Expression/BrainSpan/20190508BrainSpanGeneExpression.csv"
sfari_path = "/lab01/Projects/Lionel_Projects/biodatabases/SFARI/SFARI-Gene_genes_08-19-2024release_10-06-2024export.csv"
cao_ndd_path = "/lab01/Projects/Lionel_Projects/biodatabases/Cao_NDD_Candidate_Genes.tsv"

brainspan_df = pd.read_csv(brainspan_path, sep="\t", comment="#")
sfari_df = pd.read_csv(sfari_path, sep=",", comment="#")
cao_ndd_df = pd.read_csv(cao_ndd_path, sep="\t", comment="#")

# Prepend ENSEMBL prefix to BrainSpan gene IDs to match the format in the gene index, enabling cross-referencing during ingestion.
brainspan_df = brainspan_df.copy()
brainspan_df['ensembl_gene_id_full'] = brainspan_df['ensembl_gene_id'].map(lambda x: f"ENSEMBL:{x}")

brainspan_res, brainspan_unmapped = await ann.ingest_annotation(brainspan_df, "brainspan", gene_id_column=["geneSymbol", "ensembl_gene_id_full"], taxon="human", override=False, case_insensitive=True)
sfari_res, sfari_unmapped = await ann.ingest_annotation(sfari_df, "sfari", gene_id_column="gene-symbol", taxon="human", override=False, case_insensitive=True)
cao_ndd_res, cao_ndd_unmapped = await ann.ingest_annotation(cao_ndd_df, "cao_ndd", gene_id_column="hgnc_id", taxon="human", override=False, case_insensitive=True)

# print("BrainSpan Ingestion Result:", brainspan_res)
# print("SFARI Ingestion Result:", sfari_res)
# print("Cao NDD Ingestion Result:", cao_ndd_res)

In [ ]:
# External annotations join by name, contributing `panel.`-prefixed columns.
test = await ann.annotate(["TP53", "BRCA1", "EGFR"], AGRDataset.ALLELES, "brainspan", "sfari", "cao_ndd", taxon="human")
# display(test)
display(test.loc[:, test.columns.str.startswith("has.") | test.columns.str == 'GeneId'])

In [ ]:
# from preprocess import resolve_cache_dir, _load_gene_records
# from annotator import Annotator

# from pathlib import Path
# import pandas as pd

# cache_dir = Path("/lab01/Projects/Lionel_Projects/alliance_wrapper/cache")
# ann = Annotator(cache_dir)
# records = await _load_gene_records(
#     cache_dir, refresh=False, client=ann._client, downloader=ann._downloader
# )

# records

In [ ]:
# id_columns = {
#     "GeneId":              "PRIMARY_ID",
#     "GeneSecondaryIds":    "SECONDARY_ID",
#     "GeneSymbol":          "OFFICIAL_SYMBOL",
#     "GeneSynonyms":        "SYNONYM",
#     "GeneCrossReferences": "CROSS_REFERENCE",
# }

In [ ]:
# lookup = (
#     records[list(id_columns.keys())]
#     .rename_axis("row")
#     .reset_index() # reset index to have a column for row numbers
#     .melt(id_vars="row", var_name="kind", value_name="key")
#     .dropna(subset=["key"]) # drop rows where key is NaN
#     .assign(key=lambda df: df["key"].str.split("|")) # split the key column by "|"
#     .explode("key", ignore_index=True) # explode the list of keys into separate rows
# )
# lookup["kind"] = lookup["kind"].map(id_columns) # map the kind to the new column names

In [ ]:
# lookup